# Cheatsheet Praktikum — Klasifikasi Teks dengan PyTorch

**IF5153 Advanced NLP** · pendamping `cheatsheet_klasifikasi_teks.ipynb` (yang berisi TF-IDF +
classical ML). Notebook ini khusus jalur neural: slide 02b hal. 10 (**CNN**, Kim 2014),
hal. 11 (**RNN/LSTM**, Nowak 2017), dan hal. 12 (**word2vec**).

Memakai data yang sama di folder `data/`, terutama `data/split/train.csv` dan `data/split/test.csv`.

**Kebutuhan:** `torch`, `scikit-learn`, `pandas`, `numpy`, `gensim` (untuk §8).
Semua sudah terpasang; tidak ada model yang perlu diunduh — seluruh notebook jalan offline.

```bash
jupyter lab cheatsheet_pytorch.ipynb        # jalankan dari folder lat-prak1
```

### Daftar isi

| § | Isi | Status |
|---|---|---|
| 1 | sklearn vs PyTorch: apa yang hilang, apa yang harus ditulis sendiri | — |
| 2 | Teks → tensor: vocab, numericalization, padding, DataLoader | **wajib** |
| 3 | Model 1: Embedding + mean pooling (baseline paling sederhana) | **wajib** |
| 4 | Anatomi training loop: 6 baris yang tidak boleh salah | **wajib** |
| 5 | Evaluasi + pembanding TF-IDF | **wajib** |
| 6 | Model 2: CNN for text (slide 02b hal. 10) | opsional |
| 7 | Model 3: LSTM (slide 02b hal. 11) | opsional |
| 8 | Pretrained embedding word2vec (slide 02b hal. 12) | opsional |
| 8b | **Alur yang dipakai: pilih satu arsitektur** (yang di-copy) | **wajib** |
| 9 | Fine-tune BERT/IndoBERT — kode referensi (tidak dijalankan) | opsional |
| 10 | Error khas PyTorch & solusinya | — |
| 11 | Checklist + tabel padanan sklearn ↔ PyTorch | — |

### Wajib vs opsional

Pipeline PyTorch paling minim yang sudah sah: **vocab → tensor → DataLoader → model → training
loop → evaluasi**. §2–§5 adalah rangkaian itu dan tidak bisa dipotong. §6–§9 mengganti bagian
*model* saja; §2, §4, §5 dipakai ulang tanpa perubahan.

> **Peringatan jujur:** pada data sekecil praktikum (2.400 dokumen latih), **TF-IDF + Logistic
> Regression biasanya mengalahkan neural network apa pun yang dilatih dari nol.** Itu bukan
> tanda kodemu salah — neural butuh data jauh lebih banyak, atau embedding pretrained (§8/§9).
> §5 membuktikan ini dengan angka. Kalau di laporan kamu bisa menjelaskan *kenapa*, itu nilai plus.

---
## §1 · sklearn vs PyTorch

Di sklearn, `.fit()` menyembunyikan enam hal. Di PyTorch semuanya kamu tulis sendiri:

| Yang di sklearn otomatis | Di PyTorch jadi tanggung jawabmu |
|---|---|
| `TfidfVectorizer` mengubah teks → angka | bangun **vocab** sendiri, ubah token → **indeks**, lalu **padding** |
| batching | `Dataset` + `DataLoader` + `collate_fn` |
| loop iterasi & konvergensi | **training loop** manual (epoch, batch, backward, step) |
| pemilihan loss | pilih sendiri (`CrossEntropyLoss` untuk klasifikasi) |
| optimasi | pilih sendiri (`Adam`, `SGD`) + `zero_grad()`/`step()` |
| CPU/GPU | pindahkan model **dan** data ke `device` secara eksplisit |

**Kapan neural layak dipakai:**

| Situasi | Pilih |
|---|---|
| < 10 ribu dokumen, fitur kata sudah informatif | **TF-IDF + LinearSVC/LogReg** — hampir selalu menang |
| Puluhan ribu+ dokumen, urutan kata penting | CNN / LSTM |
| Butuh akurasi terbaik, ada GPU, boleh unduh model | fine-tune BERT/IndoBERT |
| Tugasnya "tunjukkan kamu paham neural net" | apa pun di notebook ini, ukurannya kecil saja |

---
## §2 · Teks → tensor  ·  `WAJIB`

Empat langkah yang menggantikan satu baris `TfidfVectorizer`:

1. **Tokenisasi** — teks jadi daftar token.
2. **Vocabulary** — petakan token → indeks bilangan bulat. Dibangun **hanya dari data latih**;
   token yang belum pernah dilihat jadi `<unk>`. Indeks 0 dicadangkan untuk `<pad>`.
3. **Numericalization** — daftar token jadi daftar indeks, dipotong pada panjang maksimum.
4. **Padding + batching** — kalimat dalam satu batch harus sama panjang → `pad_sequence`,
   sambil menyimpan **panjang asli** tiap kalimat (dipakai LSTM dan mean pooling).

In [1]:
import re, math, random, time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence

SEED = 42


def set_seed(s=SEED):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)


set_seed()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)
if device.type == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))

torch 2.6.0+cu124 | device: cuda
gpu: NVIDIA GeForce RTX 3060 Laptop GPU


In [2]:
DATA_DIR = Path("data")
LABELS = {0: "negatif", 4: "positif"}

def baca(path):
    df = pd.read_csv(path, encoding="utf-8")
    df = df.rename(columns={"target": "label"})[["text", "label"]]
    df["label"] = df["label"].map(LABELS).fillna(df["label"])
    df["text"] = df["text"].astype(str).str.strip()
    return df[df["text"].str.len() > 2].drop_duplicates(subset=["text"]).reset_index(drop=True)


train_full = baca(DATA_DIR / "split" / "train.csv")
test        = baca(DATA_DIR / "split" / "test.csv")

# validation dipotong dari TRAIN -- test hanya disentuh di akhir
from sklearn.model_selection import train_test_split
train, val = train_test_split(train_full, test_size=0.15, random_state=SEED,
                              stratify=train_full["label"])
print("train:", len(train), "| val:", len(val), "| test:", len(test))
print(train["label"].value_counts().to_dict())

train: 2040 | val: 360 | test: 800
{'positif': 1020, 'negatif': 1020}


In [3]:
def tokenize(teks):
    teks = teks.lower()
    teks = re.sub(r"http\S+|www\.\S+", " urltoken ", teks)
    teks = re.sub(r"@\w+", " usertoken ", teks)
    teks = re.sub(r"\d+", " numtoken ", teks)
    return re.findall(r"[a-z]+", teks)


PAD, UNK = 0, 1


def bangun_vocab(daftar_teks, min_freq=2, max_size=20000):
    c = Counter(t for teks in daftar_teks for t in tokenize(teks))
    kata = [w for w, n in c.most_common(max_size) if n >= min_freq]
    itos = ["<pad>", "<unk>"] + kata
    stoi = {w: i for i, w in enumerate(itos)}
    return stoi, itos


# PENTING: vocab dibangun dari TRAIN saja. Kalau ikut melihat val/test -> data leakage.
stoi, itos = bangun_vocab(train["text"])
print("ukuran vocab:", len(itos))
print("10 kata tersering:", itos[2:12])

MAX_LEN = 48


def encode(teks):
    ids = [stoi.get(t, UNK) for t in tokenize(teks)][:MAX_LEN]
    return ids or [UNK]                    # jangan sampai kosong


contoh = train["text"].iloc[0]
print("\nteks  :", contoh[:70])
print("token :", tokenize(contoh)[:12])
print("ids   :", encode(contoh)[:12])

ukuran vocab: 1773
10 kata tersering: ['i', 'usertoken', 'to', 'the', 'a', 'my', 'numtoken', 'you', 'and', 'it']

teks  : @special4u2 Get 100 followers a day using www.tweeterfollow.com Once y
token : ['usertoken', 'get', 'numtoken', 'followers', 'a', 'day', 'using', 'urltoken', 'once', 'you', 'add', 'everyone']
ids   : [3, 38, 8, 275, 6, 34, 427, 41, 339, 9, 428, 230]


In [4]:
KELAS = sorted(train["label"].unique())          # ['negatif', 'positif']
label2id = {c: i for i, c in enumerate(KELAS)}
print("pemetaan label:", label2id)


class DatasetTeks(Dataset):
    def __init__(self, df):
        self.ids    = [encode(t) for t in df["text"]]
        self.labels = [label2id[l] for l in df["label"]]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return torch.tensor(self.ids[i], dtype=torch.long), self.labels[i]


MIN_LEN = 5                                   # CNN butuh panjang >= ukuran kernel terbesar


def collate(batch):
    urutan, label = zip(*batch)
    panjang = torch.tensor([len(x) for x in urutan], dtype=torch.long)
    padded = pad_sequence(urutan, batch_first=True, padding_value=PAD)
    if padded.size(1) < MIN_LEN:              # padding tambahan kalau batch-nya pendek semua
        padded = F.pad(padded, (0, MIN_LEN - padded.size(1)), value=PAD)
    return padded, panjang, torch.tensor(label, dtype=torch.long)


BATCH = 64
dl_train = DataLoader(DatasetTeks(train), batch_size=BATCH, shuffle=True,  collate_fn=collate)
dl_val   = DataLoader(DatasetTeks(val),   batch_size=BATCH, shuffle=False, collate_fn=collate)
dl_test  = DataLoader(DatasetTeks(test),  batch_size=BATCH, shuffle=False, collate_fn=collate)

xb, lb, yb = next(iter(dl_train))
print("bentuk batch  :", tuple(xb.shape), "= (batch, panjang terpanjang di batch ini)")
print("panjang asli  :", lb[:8].tolist())
print("label         :", yb[:8].tolist())
print("baris pertama :", xb[0][:15].tolist(), "   <- angka 0 adalah <pad>")

pemetaan label: {'negatif': 0, 'positif': 1}
bentuk batch  : (64, 28) = (batch, panjang terpanjang di batch ini)
panjang asli  : [5, 4, 16, 9, 15, 18, 6, 20]
label         : [1, 0, 0, 1, 1, 1, 1, 1]
baris pertama : [3, 100, 59, 15, 9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]    <- angka 0 adalah <pad>


> **Kenapa `shuffle=True` hanya di train?** Urutan batch yang acak mencegah model belajar dari
> urutan data. Pada val/test urutan tidak berpengaruh dan `shuffle=False` membuat hasil prediksi
> sejajar dengan urutan dataframe — memudahkan analisis kesalahan.

---
## §3 · Model 1: Embedding + mean pooling  ·  `WAJIB`

Model neural paling sederhana untuk teks (gaya fastText). Alurnya:

```
ids (batch, panjang)
  → nn.Embedding        → (batch, panjang, dim)   tiap kata jadi vektor
  → rata-rata token asli → (batch, dim)           satu vektor per dokumen
  → nn.Linear           → (batch, jumlah_kelas)   skor tiap kelas (logit)
```

Rata-ratanya **harus mengabaikan `<pad>`** — kalau tidak, kalimat pendek yang banyak padding
akan terlarutkan nilainya. Itu gunanya `mask`.

Ini juga persis konsep "dokumen = satu vektor" di slide 02b, bedanya vektor katanya **dilatih**
bersama classifier, bukan berupa skor TF-IDF.

In [5]:
class MeanPool(nn.Module):
    def __init__(self, n_vocab, dim=100, n_kelas=2, dropout=0.3):
        super().__init__()
        self.emb = nn.Embedding(n_vocab, dim, padding_idx=PAD)   # padding_idx -> vektor <pad> tetap nol
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(dim, n_kelas)

    def forward(self, x, panjang):
        e = self.emb(x)                                  # (B, L, D)
        mask = (x != PAD).unsqueeze(-1).float()          # (B, L, 1)
        rata = (e * mask).sum(1) / mask.sum(1).clamp(min=1)   # (B, D)
        return self.fc(self.drop(rata))                  # (B, n_kelas) -- LOGIT, bukan probabilitas


set_seed()
model = MeanPool(len(itos)).to(device)
print(model)
print("jumlah parameter:", sum(p.numel() for p in model.parameters()if p.requires_grad))

with torch.no_grad():
    keluaran = model(xb.to(device), lb.to(device))
print("bentuk keluaran:", tuple(keluaran.shape), "-> (batch, jumlah kelas)")

MeanPool(
  (emb): Embedding(1773, 100, padding_idx=0)
  (drop): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=100, out_features=2, bias=True)
)
jumlah parameter: 177502


bentuk keluaran: (64, 2) -> (batch, jumlah kelas)


> `nn.EmbeddingBag` bisa melakukan mean pooling sekaligus dan sedikit lebih cepat, tapi
> antarmukanya memakai `offsets` dan tidak dipakai ulang oleh CNN/LSTM. Versi manual di atas
> memakai bentuk tensor yang sama untuk ketiga model, jadi §2, §4, dan §5 tidak perlu diubah.

---
## §4 · Anatomi training loop  ·  `WAJIB`

Enam baris di dalam loop, urutannya tidak boleh salah:

```python
optimizer.zero_grad()      # 1. hapus gradien batch sebelumnya  <- lupa ini = gradien menumpuk
logits = model(x, panjang) # 2. forward
loss = criterion(logits,y) # 3. hitung loss  (CrossEntropyLoss makan LOGIT, bukan softmax)
loss.backward()            # 4. backward: hitung gradien
optimizer.step()           # 5. perbarui bobot
```

Plus dua sakelar mode yang sering terlupa:

- `model.train()` — mengaktifkan dropout & batchnorm. Dipakai saat melatih.
- `model.eval()` + `torch.no_grad()` — mematikan dropout dan berhenti menyimpan graf gradien.
  Dipakai saat evaluasi. Lupa `no_grad()` membuat memori membengkak dan lambat; lupa `eval()`
  membuat skor validasi acak karena dropout masih aktif.

In [6]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix


@torch.no_grad()
def prediksi(model, loader):
    model.eval()                                  # matikan dropout
    semua_pred, semua_y = [], []
    for x, panjang, y in loader:
        logits = model(x.to(device), panjang.to(device))
        semua_pred.append(logits.argmax(1).cpu())
        semua_y.append(y)
    return torch.cat(semua_pred).numpy(), torch.cat(semua_y).numpy()


def latih(model, dl_train, dl_val, epochs=6, lr=1e-3, bobot_kelas=None, diam=False):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss(weight=bobot_kelas)   # bobot_kelas untuk data timpang
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    riwayat, terbaik, state_terbaik = [], -1, None
    for ep in range(1, epochs + 1):
        model.train()                              # aktifkan dropout
        total, mulai = 0.0, time.time()
        for x, panjang, y in dl_train:
            x, panjang, y = x.to(device), panjang.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x, panjang), y)
            loss.backward()
            optimizer.step()
            total += loss.item() * y.size(0)

        pred, ytrue = prediksi(model, dl_val)
        f1 = f1_score(ytrue, pred, average="macro")
        riwayat.append((ep, total / len(dl_train.dataset), f1))
        if f1 > terbaik:                           # simpan bobot terbaik menurut VALIDASI
            terbaik = f1
            state_terbaik = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if not diam:
            print(f"  epoch {ep}  loss {total/len(dl_train.dataset):.4f}"
                  f"  val_f1 {f1:.4f}  ({time.time()-mulai:.1f}s)")

    model.load_state_dict(state_terbaik)           # kembalikan ke epoch terbaik
    return model, riwayat


set_seed()
model = MeanPool(len(itos))
model, riwayat = latih(model, dl_train, dl_val, epochs=6)

  epoch 1  loss 0.7063  val_f1 0.5300  (0.2s)
  epoch 2  loss 0.6747  val_f1 0.5833  (0.1s)
  epoch 3  loss 0.6659  val_f1 0.5994  (0.1s)


  epoch 4  loss 0.6517  val_f1 0.6075  (0.1s)
  epoch 5  loss 0.6398  val_f1 0.6288  (0.1s)
  epoch 6  loss 0.6320  val_f1 0.6328  (0.1s)


> **`early stopping` versi sederhana:** simpan `state_dict` pada epoch dengan skor validasi
> terbaik, lalu kembalikan di akhir. Tanpa ini, model yang kamu evaluasi adalah model epoch
> terakhir — yang sering sudah mulai overfit. Perhatikan `loss` yang terus turun sementara
> `val_f1` mulai mandek: itulah tandanya.

---
## §5 · Evaluasi + pembanding TF-IDF  ·  `WAJIB`

Metriknya sama persis dengan cheatsheet klasik — `classification_report` dari sklearn menerima
array numpy biasa, jadi tidak ada yang khusus PyTorch di sini.

In [7]:
pred, ytrue = prediksi(model, dl_test)
print(classification_report(ytrue, pred, target_names=KELAS, zero_division=0))
print(pd.DataFrame(confusion_matrix(ytrue, pred),
                   index=["true_" + c for c in KELAS],
                   columns=["pred_" + c for c in KELAS]).to_string())

              precision    recall  f1-score   support

     negatif       0.64      0.63      0.63       400
     positif       0.63      0.64      0.64       400

    accuracy                           0.64       800
   macro avg       0.64      0.64      0.63       800
weighted avg       0.64      0.64      0.63       800



              pred_negatif  pred_positif
true_negatif           252           148
true_positif           144           256


In [8]:
# Pembanding: TF-IDF + Logistic Regression pada data yang sama persis
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

klasik = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
                   ("clf", LogisticRegression(max_iter=1000))]).fit(train["text"], train["label"])
pred_klasik = klasik.predict(test["text"])

f1_neural = f1_score(ytrue, pred, average="macro")
f1_klasik = f1_score(test["label"].map(label2id), pd.Series(pred_klasik).map(label2id),
                     average="macro")
print(f"MeanPool (PyTorch)        f1_macro = {f1_neural:.3f}")
print(f"TF-IDF + LogReg (sklearn) f1_macro = {f1_klasik:.3f}")
print("\nSelisih:", round(f1_klasik - f1_neural, 3),
      "-> positif berarti model klasik menang")

MeanPool (PyTorch)        f1_macro = 0.635
TF-IDF + LogReg (sklearn) f1_macro = 0.677

Selisih: 0.042 -> positif berarti model klasik menang


---
## §6 · Model 2: CNN for text  ·  `OPSIONAL — alternatif arsitektur`  ·  slide 02b hal. 10

Arsitektur Yoon Kim (2014). Idenya: **filter konvolusi selebar n kata = detektor n-gram yang
dipelajari**. Filter lebar 3 belajar mengenali frasa 3 kata di mana pun posisinya, lalu
**max-pooling over time** mengambil sinyal terkuat dari seluruh kalimat.

```
(B, L) → Embedding → (B, L, D) → transpose → (B, D, L)
       → Conv1d k=3 → ReLU → max-pool → (B, n_filter)  ┐
       → Conv1d k=4 → ReLU → max-pool → (B, n_filter)  ├→ concat → dropout → Linear → (B, kelas)
       → Conv1d k=5 → ReLU → max-pool → (B, n_filter)  ┘
```

Hubungannya dengan cheatsheet klasik: ini versi *learnable* dari `ngram_range=(3,5)` pada
`TfidfVectorizer` — bedanya n-gram di sini tidak dicocokkan persis, melainkan dikenali secara
semantik lewat vektor kata.

In [9]:
class CNNTeks(nn.Module):
    def __init__(self, n_vocab, dim=100, n_filter=100, kernel=(3, 4, 5), n_kelas=2, dropout=0.5):
        super().__init__()
        self.emb = nn.Embedding(n_vocab, dim, padding_idx=PAD)
        self.convs = nn.ModuleList([nn.Conv1d(dim, n_filter, k) for k in kernel])
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(n_filter * len(kernel), n_kelas)

    def forward(self, x, panjang=None):
        e = self.emb(x).transpose(1, 2)                    # (B, D, L) -- Conv1d butuh channel di tengah
        fitur = [F.relu(conv(e)).max(dim=2).values for conv in self.convs]   # max-pool over time
        return self.fc(self.drop(torch.cat(fitur, dim=1)))


set_seed()
cnn, _ = latih(CNNTeks(len(itos)), dl_train, dl_val, epochs=6)
pred_cnn, _ = prediksi(cnn, dl_test)
print("\nCNN f1_macro (test):", round(f1_score(ytrue, pred_cnn, average="macro"), 3))

  epoch 1  loss 0.7414  val_f1 0.6100  (0.7s)
  epoch 2  loss 0.5858  val_f1 0.6593  (0.1s)


  epoch 3  loss 0.5146  val_f1 0.6819  (0.2s)
  epoch 4  loss 0.4246  val_f1 0.7025  (0.2s)


  epoch 5  loss 0.3548  val_f1 0.6972  (0.1s)
  epoch 6  loss 0.2992  val_f1 0.6972  (0.1s)

CNN f1_macro (test): 0.655


---
## §7 · Model 3: LSTM  ·  `OPSIONAL — alternatif arsitektur`  ·  slide 02b hal. 11

RNN membaca kalimat **berurutan** dan membawa memori; LSTM menambahkan gerbang supaya memori
bertahan lebih lama. Berbeda dari BoW dan CNN, urutan kata dipakai sepenuhnya — inilah jawaban
paling langsung atas kelemahan #3 spam word list di slide 02a hal. 5.

**`pack_padded_sequence` itu wajib.** Tanpa itu, LSTM ikut memproses token `<pad>`, sehingga
hidden state terakhir mencerminkan padding, bukan akhir kalimat sebenarnya.

In [10]:
class LSTMTeks(nn.Module):
    def __init__(self, n_vocab, dim=100, hidden=128, n_kelas=2, dropout=0.3, bidirectional=True):
        super().__init__()
        self.emb = nn.Embedding(n_vocab, dim, padding_idx=PAD)
        self.lstm = nn.LSTM(dim, hidden, batch_first=True, bidirectional=bidirectional)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden * (2 if bidirectional else 1), n_kelas)
        self.bi = bidirectional

    def forward(self, x, panjang):
        e = self.emb(x)
        packed = pack_padded_sequence(e, panjang.cpu(), batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(packed)                  # h: (n_arah, B, hidden)
        h = torch.cat([h[-2], h[-1]], dim=1) if self.bi else h[-1]
        return self.fc(self.drop(h))


set_seed()
lstm, _ = latih(LSTMTeks(len(itos)), dl_train, dl_val, epochs=6)
pred_lstm, _ = prediksi(lstm, dl_test)
print("\nLSTM f1_macro (test):", round(f1_score(ytrue, pred_lstm, average="macro"), 3))

  epoch 1  loss 0.6734  val_f1 0.6721  (0.5s)


  epoch 2  loss 0.6172  val_f1 0.6832  (0.4s)


  epoch 3  loss 0.5526  val_f1 0.6867  (0.4s)


  epoch 4  loss 0.4630  val_f1 0.6830  (0.4s)


  epoch 5  loss 0.3637  val_f1 0.6743  (0.4s)


  epoch 6  loss 0.2536  val_f1 0.6914  (0.4s)

LSTM f1_macro (test): 0.651


---
## §8 · Pretrained embedding word2vec  ·  `OPSIONAL`  ·  slide 02b hal. 12

Sejauh ini `nn.Embedding` diinisialisasi **acak** dan dilatih dari nol — dengan 2 ribu dokumen,
vektor katanya tidak sempat jadi bermakna. Solusi slide 02b: pakai **word2vec** yang dilatih
pada korpus besar, lalu masukkan sebagai bobot awal `nn.Embedding`.

Di sini word2vec dilatih pada korpus latih itu sendiri (cepat, offline, tanpa unduh). Di praktikum
sungguhan kamu akan memuat vektor terlatih (`GoogleNews`, `fasttext-id`, dsb) lewat
`gensim.downloader` — konsep dan kodenya sama, tinggal ganti sumber vektornya.

**Dua pilihan penting:** `freeze=True` (vektor tidak ikut dilatih — aman untuk data kecil) vs
`freeze=False` (ikut disetel — perlu data lebih banyak).

In [11]:
from gensim.models import Word2Vec

kalimat_token = [tokenize(t) for t in train["text"]]
set_seed()
w2v = Word2Vec(sentences=kalimat_token, vector_size=100, window=5, min_count=2,
               sg=1, epochs=10, workers=1, seed=SEED)      # sg=1 -> skip-gram (slide 02b)
print("kata di model word2vec:", len(w2v.wv))

# Klaim utama slide 02b hal. 4: "kata mirip punya vektor berdekatan"
for kata in ["good", "day", "love"]:
    if kata in w2v.wv:
        mirip = [w for w, _ in w2v.wv.most_similar(kata, topn=5)]
        print(f"  paling mirip dengan '{kata}':", mirip)
print("\nHasilnya jelek, dan itu WAJAR: word2vec di sini dilatih pada 2 ribu tweet saja.")
print("Kualitas embedding butuh korpus jutaan kalimat -- itulah alasan orang memakai")
print("vektor pretrained (GoogleNews, fastText-id) alih-alih melatih sendiri.")

kata di model word2vec: 1771
  paling mirip dengan 'good': ['haha', 'doing', 'sounds', 'true', 'pretty']
  paling mirip dengan 'day': ['last', 'in', 'from', 'night', 'the']
  paling mirip dengan 'love': ['think', 'thank', 'if', 'would', 'hope']

Hasilnya jelek, dan itu WAJAR: word2vec di sini dilatih pada 2 ribu tweet saja.
Kualitas embedding butuh korpus jutaan kalimat -- itulah alasan orang memakai
vektor pretrained (GoogleNews, fastText-id) alih-alih melatih sendiri.


In [12]:
# Susun matriks embedding sejajar dengan vocab kita
dim = w2v.wv.vector_size
matriks = np.random.normal(0, 0.1, (len(itos), dim)).astype("float32")
matriks[PAD] = 0
ketemu = 0
for i, kata in enumerate(itos):
    if kata in w2v.wv:
        matriks[i] = w2v.wv[kata]
        ketemu += 1
print(f"kata yang dapat vektor pretrained: {ketemu}/{len(itos)} ({ketemu/len(itos):.0%})")


class MeanPoolPretrained(MeanPool):
    def __init__(self, bobot, freeze=True, n_kelas=2, dropout=0.3):
        super().__init__(bobot.shape[0], bobot.shape[1], n_kelas, dropout)
        self.emb = nn.Embedding.from_pretrained(torch.tensor(bobot), freeze=freeze,
                                                padding_idx=PAD)


hasil_w2v = {}
for freeze in [True, False]:
    set_seed()
    m, _ = latih(MeanPoolPretrained(matriks, freeze=freeze), dl_train, dl_val,
                 epochs=6, diam=True)
    p, _ = prediksi(m, dl_test)
    hasil_w2v[freeze] = f1_score(ytrue, p, average="macro")
    print(f"word2vec freeze={str(freeze):5s} -> f1_macro(test) = {hasil_w2v[freeze]:.3f}")

kata yang dapat vektor pretrained: 1771/1773 (100%)


word2vec freeze=True  -> f1_macro(test) = 0.571


word2vec freeze=False -> f1_macro(test) = 0.715


---
## §8b · Alur yang dipakai — pilih SATU arsitektur  ·  `INI YANG DI-COPY`

§3, §6, §7, dan §8 masing-masing melatih modelnya sendiri **dengan sengaja**: tujuannya
membandingkan keempatnya di tabel §11. Itu bukan alur yang kamu pakai di praktikum.

Kalau kamu cuma butuh satu model, sel di bawah ini yang di-copy. Empat pilihan saling
menggantikan — satu aktif, sisanya komentar. Semua memakai `dl_train` / `dl_val` / `dl_test`
dari §2, `latih()` dari §4, dan `prediksi()` dari §4 tanpa perubahan.

In [13]:
# =============================================================
# PILIH SATU -- uncomment yang dipakai, comment yang lain
# =============================================================
set_seed()
model_pilih = MeanPool(len(itos))                              # 1) paling sederhana & cepat  <- aktif
# model_pilih = CNNTeks(len(itos))                             # 2) CNN, Kim 2014 (02b hal. 10)
# model_pilih = LSTMTeks(len(itos))                            # 3) LSTM dua arah (02b hal. 11)
# model_pilih = MeanPoolPretrained(matriks, freeze=False)      # 4) + word2vec (butuh sel §8 sudah jalan)

# Training + evaluasi -- bagian ini TIDAK berubah apa pun arsitekturnya
model_pilih, _ = latih(model_pilih, dl_train, dl_val, epochs=6)

pred_pilih, y_pilih = prediksi(model_pilih, dl_test)
print()
print(classification_report(y_pilih, pred_pilih, target_names=KELAS, zero_division=0))
print("f1_macro:", round(f1_score(y_pilih, pred_pilih, average="macro"), 3))

  epoch 1  loss 0.7063  val_f1 0.5300  (0.1s)
  epoch 2  loss 0.6747  val_f1 0.5833  (0.1s)


  epoch 3  loss 0.6659  val_f1 0.5994  (0.1s)


  epoch 4  loss 0.6517  val_f1 0.6075  (0.1s)
  epoch 5  loss 0.6398  val_f1 0.6288  (0.1s)


  epoch 6  loss 0.6320  val_f1 0.6328  (0.1s)

              precision    recall  f1-score   support

     negatif       0.64      0.63      0.63       400
     positif       0.63      0.64      0.64       400

    accuracy                           0.64       800
   macro avg       0.64      0.64      0.63       800
weighted avg       0.64      0.64      0.63       800

f1_macro: 0.635


> **Yang perlu diubah kalau ganti arsitektur: hanya satu baris.** Loader, training loop, dan
> evaluasi dipakai ulang apa adanya. Itulah gunanya memisahkan `latih()` dan `prediksi()` jadi
> fungsi sejak §4 — kalau ditulis inline di tiap section, ganti model berarti menyalin ulang
> seluruh loop.
>
> Hyperparameter tiap arsitektur diatur lewat argumen konstruktornya:
> `CNNTeks(len(itos), dim=128, n_filter=150, kernel=(2,3,4))`,
> `LSTMTeks(len(itos), hidden=256, bidirectional=False)`.

---
## §9 · Fine-tune BERT / IndoBERT  ·  `OPSIONAL, tidak dijalankan di sini`

Sel di bawah sengaja ditulis sebagai **teks**, bukan kode yang dieksekusi, karena butuh mengunduh
model (±500 MB) dan idealnya GPU. Simpan sebagai referensi kalau besok ternyata diminta.

```python
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import torch

NAMA = "indobenchmark/indobert-base-p1"     # bahasa Indonesia
# NAMA = "distilbert-base-uncased"          # bahasa Inggris, lebih ringan

tok = AutoTokenizer.from_pretrained(NAMA)
model = AutoModelForSequenceClassification.from_pretrained(NAMA, num_labels=2)

class DS(torch.utils.data.Dataset):
    def __init__(self, teks, label):
        self.enc = tok(list(teks), truncation=True, padding=True, max_length=128)
        self.label = list(label)
    def __len__(self):  return len(self.label)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.label[i])
        return item

args = TrainingArguments(output_dir="out", num_train_epochs=2, per_device_train_batch_size=16,
                         learning_rate=2e-5, eval_strategy="epoch", report_to="none")
Trainer(model=model, args=args,
        train_dataset=DS(train["text"], train["label"].map(label2id)),
        eval_dataset=DS(val["text"],   val["label"].map(label2id))).train()
```

Catatan praktis: `datasets` belum terpasang di laptopmu (`pip install datasets`), tapi contoh di
atas sengaja memakai `Dataset` bawaan PyTorch supaya tidak membutuhkannya. Learning rate untuk
fine-tuning transformer **jauh lebih kecil** (2e-5), jangan dipakai 1e-3 seperti model dari nol.

---
## §10 · Error khas PyTorch & solusinya

| Pesan / gejala | Penyebab | Solusi |
|---|---|---|
| `Expected all tensors to be on the same device` | model di GPU, data di CPU (atau sebaliknya) | `x.to(device)` untuk **semua** tensor, dan `model.to(device)` |
| `expected scalar type Long but found Float` | label atau indeks kata bertipe float | `torch.tensor(..., dtype=torch.long)` |
| `Target N is out of bounds` | jumlah kelas di `Linear` tidak cocok dengan label | keluaran `Linear` = jumlah kelas; label harus 0..k-1 |
| Loss tidak turun sama sekali | lupa `optimizer.zero_grad()` atau `loss.backward()` | ikuti urutan 5 baris di §4 |
| Loss `nan` | learning rate terlalu besar | turunkan ke `1e-3`/`1e-4`, atau `clip_grad_norm_` |
| Skor validasi acak / naik-turun liar | lupa `model.eval()` saat evaluasi | pakai `@torch.no_grad()` + `model.eval()` |
| Memori habis saat evaluasi | lupa `torch.no_grad()` | bungkus evaluasi dengan `no_grad` |
| `pack_padded_sequence` error soal urutan | batch tidak terurut menurun | `enforce_sorted=False` |
| Akurasi bagus di train, jelek di test | overfit (biasa pada data kecil) | dropout, `weight_decay`, epoch lebih sedikit, embedding pretrained |
| Hasil berbeda tiap dijalankan | seed belum diset | `set_seed()` sebelum membuat model **dan** sebelum melatih |
| CNN error `Kernel size can't be greater than input` | kalimat lebih pendek dari kernel | padding minimum (`MIN_LEN` di §2) |

---
## §11 · Checklist + padanan sklearn ↔ PyTorch

| Kebutuhan | scikit-learn | PyTorch |
|---|---|---|
| Teks → angka | `TfidfVectorizer()` | `bangun_vocab` + `encode` + `pad_sequence` |
| Batching | otomatis | `Dataset` + `DataLoader(collate_fn=...)` |
| Model | `LogisticRegression()` | `nn.Module` dengan `forward()` |
| Latih | `.fit(X, y)` | training loop: `zero_grad → forward → loss → backward → step` |
| Prediksi | `.predict(X)` | `model.eval()`; `logits.argmax(1)` |
| Probabilitas | `.predict_proba(X)` | `F.softmax(logits, dim=1)` |
| Loss | tersembunyi | `nn.CrossEntropyLoss()` (menerima **logit**) |
| Kelas timpang | `class_weight="balanced"` | `CrossEntropyLoss(weight=bobot)` |
| Regularisasi | parameter `C` | `nn.Dropout`, `weight_decay` pada optimizer |
| Simpan model | `joblib.dump(model, ...)` | `torch.save(model.state_dict(), ...)` |
| Muat model | `joblib.load(...)` | buat model kosong lalu `load_state_dict(torch.load(...))` |

### Checklist saat praktikum

1. **Jalankan baseline sklearn dulu** (cheatsheet satunya). Itu angka pembanding wajib; tanpa itu
   kamu tidak tahu apakah neural-mu berguna.
2. **Set seed** sebelum membuat model dan sebelum melatih, supaya hasilnya bisa diulang.
3. **Bangun vocab dari train saja.** Ini kesalahan leakage yang paling sering lolos.
4. Mulai dari model **paling sederhana** (§3). Naik ke CNN/LSTM hanya kalau baseline sudah jalan.
5. Cek `loss` epoch pertama masuk akal: untuk 2 kelas seimbang, awalnya sekitar `ln(2) ≈ 0.69`.
   Kalau jauh lebih besar atau `nan`, ada yang salah sebelum training.
6. **Pilih model berdasarkan validasi**, ukur test **sekali** di akhir.
7. Laporkan `classification_report` + confusion matrix, dan jelaskan kenapa neural kalah/menang
   dibanding TF-IDF.

### Simpan & muat model

In [14]:
torch.save({"state_dict": model.state_dict(), "itos": itos, "kelas": KELAS}, "model_meanpool.pt")

ckpt = torch.load("model_meanpool.pt", map_location=device, weights_only=False)
model_baru = MeanPool(len(ckpt["itos"])).to(device)
model_baru.load_state_dict(ckpt["state_dict"])
pred_baru, _ = prediksi(model_baru, dl_test)
print("hasil model yang dimuat ulang sama:", bool((pred_baru == pred).all()))
print("f1_macro:", round(f1_score(ytrue, pred_baru, average="macro"), 3))

hasil model yang dimuat ulang sama: True
f1_macro: 0.635


> Simpan **`state_dict`**, bukan objek modelnya. Simpan juga `itos` (vocab) dan daftar kelas —
> tanpa itu model tidak bisa dipakai lagi karena indeks katanya tidak diketahui.

### Ringkasan hasil di notebook ini

In [15]:
ringkas = pd.DataFrame([
    ("TF-IDF + LogReg (sklearn)",          f1_klasik),
    ("MeanPool (embedding acak)",          f1_score(ytrue, pred, average="macro")),
    ("CNN (Kim 2014)",                     f1_score(ytrue, pred_cnn, average="macro")),
    ("LSTM bidirectional",                 f1_score(ytrue, pred_lstm, average="macro")),
    ("MeanPool + word2vec (dibekukan)",    hasil_w2v[True]),
    ("MeanPool + word2vec (ikut dilatih)", hasil_w2v[False]),
], columns=["model", "f1_macro (test)"]).sort_values("f1_macro (test)", ascending=False)
print(ringkas.round(3).to_string(index=False))
print(f"\nData latih hanya {len(train)} dokumen. Neural yang dilatih dari nol kalah dari TF-IDF;")
print("yang membalikkan keadaan adalah embedding pretrained yang ikut disetel (§8) --")
print("persis argumen slide 02b: representasi kata yang baik lebih menentukan daripada arsitektur.")

                             model  f1_macro (test)
MeanPool + word2vec (ikut dilatih)            0.715
         TF-IDF + LogReg (sklearn)            0.677
                    CNN (Kim 2014)            0.655
                LSTM bidirectional            0.651
         MeanPool (embedding acak)            0.635
   MeanPool + word2vec (dibekukan)            0.571

Data latih hanya 2040 dokumen. Neural yang dilatih dari nol kalah dari TF-IDF;
yang membalikkan keadaan adalah embedding pretrained yang ikut disetel (§8) --
persis argumen slide 02b: representasi kata yang baik lebih menentukan daripada arsitektur.
